<a href="https://colab.research.google.com/github/krishnayaswanthp-netizen/Feb-2026-Internship-Innomatics-Research-Lab/blob/main/Assignment2GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

df = pd.read_csv("twitter_training.csv", header=None)

df.columns = ['tweet_id', 'entity', 'sentiment', 'text']

df = df[['text', 'sentiment']]

df = df.dropna()

print(df.head())
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

                                                text sentiment
0  im getting on borderlands and i will murder yo...  Positive
1  I am coming to the borders and I will kill you...  Positive
2  im getting on borderlands and i will kill you ...  Positive
3  im coming on borderlands and i will murder you...  Positive
4  im getting on borderlands 2 and i will murder ...  Positive
Shape: (73996, 2)

Class Distribution:
 sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()  # safer

    cleaned_words = []
    for word in words:
        if word not in stop_words:
            cleaned_words.append(stemmer.stem(word))

    return " ".join(cleaned_words)

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


                                                text  \
0  im getting on borderlands and i will murder yo...   
1  I am coming to the borders and I will kill you...   
2  im getting on borderlands and i will kill you ...   
3  im coming on borderlands and i will murder you...   
4  im getting on borderlands 2 and i will murder ...   

                  clean_text  
0   im get borderland murder  
1           come border kill  
2     im get borderland kill  
3  im come borderland murder  
4   im get borderland murder  


Task 3: Feature Engineering

In [10]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (59196,)
Test size: (14800,)


In [11]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print("BoW shape:", X_train_bow.shape)

BoW shape: (59196, 5000)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (59196, 5000)


Task 4: Model Building

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

Task 5: Model Evaluation

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    return accuracy, precision, recall, f1

Task 6: Train Models + Compare

In [17]:
results_bow = {}

for name, model in models.items():
    results_bow[name] = evaluate_model(
        model, X_train_bow, X_test_bow, y_train, y_test
    )

print("BoW Results:")
for model, scores in results_bow.items():
    print(model, scores)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BoW Results:
Logistic Regression (0.7108108108108108, 0.7137186693983636, 0.7108108108108108, 0.7085169680378304)
Naive Bayes (0.632972972972973, 0.6340577912992278, 0.632972972972973, 0.6268473862641892)
Decision Tree (0.8185135135135135, 0.8201715075848244, 0.8185135135135135, 0.8184155629129203)


In [18]:
results_tfidf = {}

for name, model in models.items():
    results_tfidf[name] = evaluate_model(
        model, X_train_tfidf, X_test_tfidf, y_train, y_test
    )

print("\nTF-IDF Results:")
for model, scores in results_tfidf.items():
    print(model, scores)


TF-IDF Results:
Logistic Regression (0.6829054054054055, 0.6838148493905662, 0.6829054054054055, 0.6789252164363955)
Naive Bayes (0.6372972972972973, 0.6538387937968649, 0.6372972972972973, 0.6237032990084235)
Decision Tree (0.7941891891891892, 0.7958040158938086, 0.7941891891891892, 0.7938499059526711)


Task 7: Final Comparison Table

In [19]:
import pandas as pd

comparison = []

for model in models.keys():

    comparison.append([
        model, "BoW",
        *results_bow[model]
    ])

    comparison.append([
        model, "TF-IDF",
        *results_tfidf[model]
    ])

comparison_df = pd.DataFrame(
    comparison,
    columns=["Model", "Vectorizer", "Accuracy", "Precision", "Recall", "F1 Score"]
)

print(comparison_df)

                 Model Vectorizer  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression        BoW  0.710811   0.713719  0.710811  0.708517
1  Logistic Regression     TF-IDF  0.682905   0.683815  0.682905  0.678925
2          Naive Bayes        BoW  0.632973   0.634058  0.632973  0.626847
3          Naive Bayes     TF-IDF  0.637297   0.653839  0.637297  0.623703
4        Decision Tree        BoW  0.818514   0.820172  0.818514  0.818416
5        Decision Tree     TF-IDF  0.794189   0.795804  0.794189  0.793850


Task 8: Insights

1. TF-IDF performed better than Bag of Words because it assigns importance to rare words.
2. Logistic Regression achieved the highest accuracy among all models.
3. Naive Bayes performed well and is computationally efficient.
4. Decision Tree showed lower performance due to overfitting on text data.
5. Preprocessing significantly improved model performance.